# Implementación de modelos de Redes Neuronales Profundas

En este archivo, voy a poner en práctica lo aprendido a lo largo de la asignatura tanto en la parte de teoría como en la parte de práctica. La idea es crear una CNN capaz de analizar radiografías de tórax y clasificarlas en tres categorías: pacientes sanos, con neumonía bacteriana y con neumonía vírica.

**Realizado por Rodrigo Gálvez Travalja**

---

## Estructura del notebook
1. Instalación e importación de librerías
2. Comprobación de GPU
3. Descarga del dataset
4. Preprocesamiento e imágenes
5. Definición de la CNN
6. Entrenamiento y validación
7. Predicción y visualización
8. Guardado del modelo

## 1. Instalación e importación de librerías

In [1]:
%pip install torch torchvision kagglehub matplotlib pandas scikit-learn --quiet

import os
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms.functional import to_pil_image
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from collections import Counter

import kagglehub

# Añadir raíz del proyecto al path para importar desde app/
sys.path.insert(0, os.path.abspath('..'))

print('Librerías importadas correctamente.')

Note: you may need to restart the kernel to use updated packages.
Librerías importadas correctamente.


/Users/rodrigo/Library/Python/3.13/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Comprobación de GPU y configuración del device

In [2]:
# Verificar disponibilidad de CUDA/MPS y seleccionar device
print(f'Versión de PyTorch: {torch.__version__}')
print(f'CUDA disponible:    {torch.cuda.is_available()}')
print(f'MPS disponible:     {torch.backends.mps.is_available()}')

if torch.cuda.is_available():
    print(f'GPU (CUDA):         {torch.cuda.get_device_name(0)}')
    print(f'Versión CUDA:       {torch.version.cuda}')
elif torch.backends.mps.is_available():
    print(f'GPU (MPS):          Apple Silicon')

# Prioridad: CUDA (Windows/Linux con GPU) → MPS (Mac Apple Silicon) → CPU
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'\nUsando device: {device}')

Versión de PyTorch: 2.11.0
CUDA disponible:    False
MPS disponible:     True
GPU (MPS):          Apple Silicon

Usando device: mps


## 3. Descarga del dataset desde Kaggle

In [3]:
# Descarga del dataset Chest X-Ray Images (Pneumonia)
ruta = kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia')
print(f'Dataset descargado en: {ruta}')

# Rutas a los splits del dataset
ruta_datos    = os.path.join(ruta, 'chest_xray')
RUTA_TRAIN    = os.path.join(ruta_datos, 'train')
RUTA_VAL      = os.path.join(ruta_datos, 'val')
RUTA_TEST     = os.path.join(ruta_datos, 'test')   # minúsculas: Linux es case-sensitive

# Verificar que las carpetas existen
for nombre, ruta_split in [('train', RUTA_TRAIN), ('val', RUTA_VAL), ('test', RUTA_TEST)]:
    existe = os.path.isdir(ruta_split)
    print(f'  {nombre}: {ruta_split} → {"OK" if existe else "NO ENCONTRADA"}')

Dataset descargado en: /Users/rodrigo/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2
  train: /Users/rodrigo/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/train → OK
  val: /Users/rodrigo/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/val → OK
  test: /Users/rodrigo/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/test → OK


## 4. Preprocesamiento e imágenes

El dataset de Kaggle organiza las imágenes en dos carpetas: `NORMAL` y `PNEUMONIA`.
Dentro de `PNEUMONIA`, el nombre del fichero indica si es bacteriana (`bacteria`) o vírica (`virus`).
Creamos un Dataset personalizado que lee esta estructura y asigna las etiquetas correctas para la clasificación en 3 clases.

In [4]:
from app.model.architecture import PneumoniaCNN, CLASS_NAMES

# Tamaño de imagen y valores de normalización ImageNet
TAMANIO_IMAGEN = (100, 100)
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]

# Usar CLASS_NAMES de architecture.py para garantizar consistencia con el wrapper
NOMBRES_CLASES = CLASS_NAMES  # ['NORMAL', 'PNEUMONIA_BACTERIAL', 'PNEUMONIA_VIRAL']

# Transformaciones para entrenamiento (con data augmentation)
transformacion_entrenamiento = transforms.Compose([
    transforms.Resize(TAMANIO_IMAGEN),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Transformaciones para validación y test (sin augmentation, solo normalización)
transformacion_val_test = transforms.Compose([
    transforms.Resize(TAMANIO_IMAGEN),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])


class DatasetRayosX(Dataset):
    """
    Dataset personalizado para radiografías de tórax con 3 clases.
    Lee la estructura de carpetas NORMAL / PNEUMONIA del dataset de Kaggle
    y distingue neumonía bacteriana y vírica por el nombre del fichero.
    """

    MAPA_ETIQUETAS = {
        'NORMAL':              0,
        'PNEUMONIA_BACTERIAL': 1,
        'PNEUMONIA_VIRAL':     2
    }

    def __init__(self, directorio_raiz, transformacion=None):
        self.rutas_imagenes = []
        self.etiquetas      = []
        self.transformacion = transformacion

        for ruta_img in glob.glob(os.path.join(directorio_raiz, 'NORMAL', '*.jpeg')):
            self.rutas_imagenes.append(ruta_img)
            self.etiquetas.append(self.MAPA_ETIQUETAS['NORMAL'])

        for ruta_img in glob.glob(os.path.join(directorio_raiz, 'PNEUMONIA', '*.jpeg')):
            nombre = os.path.basename(ruta_img).lower()
            if 'bacteria' in nombre:
                self.rutas_imagenes.append(ruta_img)
                self.etiquetas.append(self.MAPA_ETIQUETAS['PNEUMONIA_BACTERIAL'])
            elif 'virus' in nombre:
                self.rutas_imagenes.append(ruta_img)
                self.etiquetas.append(self.MAPA_ETIQUETAS['PNEUMONIA_VIRAL'])

        print(f'Dataset cargado desde: {directorio_raiz}')
        print(f'  NORMAL:               {self.etiquetas.count(0)}')
        print(f'  PNEUMONIA_BACTERIAL:  {self.etiquetas.count(1)}')
        print(f'  PNEUMONIA_VIRAL:      {self.etiquetas.count(2)}')
        print(f'  Total:                {len(self.rutas_imagenes)}')

    def __len__(self):
        return len(self.rutas_imagenes)

    def __getitem__(self, idx):
        imagen   = Image.open(self.rutas_imagenes[idx]).convert('RGB')
        etiqueta = self.etiquetas[idx]
        if self.transformacion:
            imagen = self.transformacion(imagen)
        return imagen, etiqueta


# Crear datasets
dataset_train_completo = DatasetRayosX(RUTA_TRAIN, transformacion=transformacion_entrenamiento)
dataset_test           = DatasetRayosX(RUTA_TEST,  transformacion=transformacion_val_test)

# Dividir train en train (80%) y validación (20%) con stratify
indices_train, indices_val = train_test_split(
    list(range(len(dataset_train_completo))),
    test_size=0.2,
    stratify=dataset_train_completo.etiquetas,
    random_state=42
)

dataset_train = data.Subset(dataset_train_completo, indices_train)
dataset_val   = data.Subset(dataset_train_completo, indices_val)

# DataLoaders
loader_train = DataLoader(dataset_train, batch_size=32, shuffle=True,  num_workers=2)
loader_val   = DataLoader(dataset_val,   batch_size=32, shuffle=False, num_workers=2)
loader_test  = DataLoader(dataset_test,  batch_size=32, shuffle=False, num_workers=2)

print(f'\nBatches de entrenamiento: {len(loader_train)}')
print(f'Batches de validación:    {len(loader_val)}')
print(f'Batches de test:          {len(loader_test)}')

# Calcular pesos de clase inversamente proporcionales a su frecuencia en train
conteo_clases = Counter(dataset_train_completo.etiquetas[i] for i in indices_train)
total_train   = sum(conteo_clases.values())
pesos_clases  = torch.tensor(
    [total_train / conteo_clases[i] for i in range(len(NOMBRES_CLASES))],
    dtype=torch.float
)
print(f'\nPesos de clase: {[f"{p:.2f}" for p in pesos_clases.tolist()]}')
print(f'  (mayor peso = clase menos frecuente)')

Dataset cargado desde: /Users/rodrigo/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/train
  NORMAL:               1341
  PNEUMONIA_BACTERIAL:  2530
  PNEUMONIA_VIRAL:      1345
  Total:                5216
Dataset cargado desde: /Users/rodrigo/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/test
  NORMAL:               234
  PNEUMONIA_BACTERIAL:  242
  PNEUMONIA_VIRAL:      148
  Total:                624

Batches de entrenamiento: 131
Batches de validación:    33
Batches de test:          20

Pesos de clase: ['3.89', '2.06', '3.88']
  (mayor peso = clase menos frecuente)


## 5. Definición de la CNN

Arquitectura de 3 bloques convolucionales seguidos de capas fully connected.
Cada bloque aplica Conv2d + ReLU + MaxPool2d(2,2), reduciendo la resolución a la mitad.
Con input 100×100: 100 → 50 → 25 → 12, por lo que la capa FC recibe 128 × 12 × 12 = 18432 características.

In [5]:
# Importar PneumoniaCNN desde app/model/architecture.py
# Misma clase que usa el wrapper de inferencia → garantiza consistencia total
modelo = PneumoniaCNN(num_classes=3).to(device)
print(modelo)

total_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f'\nParámetros entrenables: {total_params:,}')

PneumoniaCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=18432, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=3, bias=True)
)

Parámetros entrenables: 2,453,059


## 6. Entrenamiento y validación

Criterio de parada: loss de entrenamiento ≤ 0.2.

In [ ]:
def entrenar_modelo(modelo, loader_train, loader_val, pesos_clases, num_epocas=20, lr=0.001):
    """
    Entrena el modelo y evalúa en validación cada época.
    Se detiene si el loss de entrenamiento baja de 0.2.

    Args:
        pesos_clases: Tensor con pesos por clase para CrossEntropyLoss (corrige desbalanceo).
    """
    optimizador = optim.RMSprop(modelo.parameters(), lr=lr)
    criterio    = nn.CrossEntropyLoss(weight=pesos_clases.to(device))

    perdidas_train,    perdidas_val     = [], []
    precisiones_train, precisiones_val  = [], []

    for epoca in range(num_epocas):

        # --- Fase de entrenamiento ---
        modelo.train()
        perdida_acum = 0.0
        correctos    = 0
        total        = 0

        for entradas, etiquetas in loader_train:
            entradas  = entradas.to(device)
            etiquetas = etiquetas.to(device)

            optimizador.zero_grad()
            salidas = modelo(entradas)
            perdida = criterio(salidas, etiquetas)
            perdida.backward()
            optimizador.step()

            perdida_acum += perdida.item()
            _, predicho   = torch.max(salidas, 1)
            correctos     += (predicho == etiquetas).sum().item()
            total         += etiquetas.size(0)

        loss_train = perdida_acum / len(loader_train)
        acc_train  = 100 * correctos / total
        perdidas_train.append(loss_train)
        precisiones_train.append(acc_train)

        # --- Fase de validación ---
        modelo.eval()
        perdida_val_acum = 0.0
        correctos_val    = 0
        total_val        = 0

        with torch.no_grad():
            for entradas, etiquetas in loader_val:
                entradas  = entradas.to(device)
                etiquetas = etiquetas.to(device)

                salidas = modelo(entradas)
                perdida = criterio(salidas, etiquetas)

                perdida_val_acum += perdida.item()
                _, predicho       = torch.max(salidas, 1)
                correctos_val    += (predicho == etiquetas).sum().item()
                total_val        += etiquetas.size(0)

        loss_val = perdida_val_acum / len(loader_val)
        acc_val  = 100 * correctos_val / total_val
        perdidas_val.append(loss_val)
        precisiones_val.append(acc_val)

        print(f'Época {epoca+1:02d}/{num_epocas} | '
              f'Loss train: {loss_train:.4f}  Acc train: {acc_train:.2f}% | '
              f'Loss val: {loss_val:.4f}  Acc val: {acc_val:.2f}%')

        if loss_train <= 0.2:
            print(f'\nCriterio de parada alcanzado en época {epoca+1}: loss train = {loss_train:.4f} ≤ 0.2')
            break

    return modelo, perdidas_train, perdidas_val, precisiones_train, precisiones_val


modelo, perdidas_train, perdidas_val, precisiones_train, precisiones_val = entrenar_modelo(
    modelo, loader_train, loader_val, pesos_clases, num_epocas=20, lr=0.001
)

## 7. Visualización de métricas de entrenamiento

In [ ]:
def mostrar_graficas(perdidas_train, perdidas_val, precisiones_train, precisiones_val):
    """Muestra las gráficas de loss y accuracy para train y validación."""
    epocas = range(1, len(perdidas_train) + 1)

    plt.figure(figsize=(14, 5))

    # Gráfica de pérdida
    plt.subplot(1, 2, 1)
    plt.plot(epocas, perdidas_train, label='Entrenamiento')
    plt.plot(epocas, perdidas_val,   label='Validación')
    plt.axhline(y=0.2, color='r', linestyle='--', label='Criterio parada (0.2)')
    plt.title('Pérdida por época')
    plt.xlabel('Época')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Gráfica de precisión
    plt.subplot(1, 2, 2)
    plt.plot(epocas, precisiones_train, label='Entrenamiento')
    plt.plot(epocas, precisiones_val,   label='Validación')
    plt.title('Precisión por época')
    plt.xlabel('Época')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()


mostrar_graficas(perdidas_train, perdidas_val, precisiones_train, precisiones_val)

## 8. Evaluación sobre el conjunto de test

Evaluación completa: accuracy global, métricas por clase (precision, recall, F1) y matriz de confusión.

In [ ]:
def evaluar_en_test(modelo, loader_test, nombres_clases):
    """Calcula accuracy, métricas por clase y matriz de confusión sobre el test set."""
    modelo.eval()
    todas_predicciones = []
    todas_etiquetas    = []

    with torch.no_grad():
        for entradas, etiquetas in loader_test:
            entradas = entradas.to(device)
            salidas  = modelo(entradas)
            _, predicho = torch.max(salidas, 1)
            todas_predicciones.extend(predicho.cpu().numpy())
            todas_etiquetas.extend(etiquetas.numpy())

    # Accuracy global
    acc = 100 * sum(p == e for p, e in zip(todas_predicciones, todas_etiquetas)) / len(todas_etiquetas)
    print(f'Accuracy en test: {acc:.2f}%\n')

    # Métricas por clase: precision, recall, F1
    print(classification_report(todas_etiquetas, todas_predicciones, target_names=nombres_clases))

    # Matriz de confusión
    cm = confusion_matrix(todas_etiquetas, todas_predicciones)
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Matriz de confusión (Test)')
    plt.colorbar()
    tick_marks = range(len(nombres_clases))
    plt.xticks(tick_marks, nombres_clases, rotation=30, ha='right')
    plt.yticks(tick_marks, nombres_clases)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha='center', va='center',
                     color='white' if cm[i, j] > cm.max() / 2 else 'black')

    plt.ylabel('Etiqueta real')
    plt.xlabel('Etiqueta predicha')
    plt.tight_layout()
    plt.show()


evaluar_en_test(modelo, loader_test, NOMBRES_CLASES)

# Mostrar algunas predicciones individuales con probabilidades
print('\n--- Ejemplos de predicciones individuales ---')
modelo.eval()
mostradas = 0
with torch.no_grad():
    for entradas, etiquetas in loader_test:
        entradas = entradas.to(device)
        salidas  = modelo(entradas)
        probs    = F.softmax(salidas, dim=1).cpu().numpy()

        for i in range(entradas.size(0)):
            if mostradas >= 5:
                break
            prob_imagen = probs[i]
            clase_pred  = NOMBRES_CLASES[np.argmax(prob_imagen)]
            clase_real  = NOMBRES_CLASES[etiquetas[i].item()]
            orden       = np.argsort(prob_imagen)[::-1]

            imagen = to_pil_image(entradas[i].cpu())
            plt.imshow(imagen, cmap='gray')
            plt.axis('off')
            acierto = '✓' if clase_pred == clase_real else '✗'
            plt.title(f'{acierto} Real: {clase_real} | Pred: {clase_pred}')
            plt.show()

            for idx in orden:
                print(f'  {NOMBRES_CLASES[idx]:<25} {prob_imagen[idx]*100:.2f}%')
            print('-' * 40)
            mostradas += 1

        if mostradas >= 5:
            break

## 9. Guardado del modelo

Guardamos el `state_dict` junto con metadatos del modelo para que el wrapper de inferencia pueda cargarlo correctamente.

In [ ]:
import os

# Ruta de guardado relativa al proyecto
RUTA_MODELO = os.path.join('..', 'models', 'model.pth')
os.makedirs(os.path.dirname(RUTA_MODELO), exist_ok=True)

# Guardar state_dict + metadatos
checkpoint = {
    'state_dict':       modelo.state_dict(),
    'num_clases':       3,
    'nombres_clases':   NOMBRES_CLASES,
    'tamanio_imagen':   TAMANIO_IMAGEN,
    'arquitectura':     'PneumoniaCNN'
}

torch.save(checkpoint, RUTA_MODELO)

# Verificar que el archivo se creó correctamente
tamanio_kb = os.path.getsize(RUTA_MODELO) / 1024
print(f'Modelo guardado en: {RUTA_MODELO}')
print(f'Tamaño del archivo: {tamanio_kb:.1f} KB')

# Verificación: cargar el modelo y hacer una predicción de prueba
print('\nVerificando carga del modelo...')
checkpoint_cargado = torch.load(RUTA_MODELO, map_location='cpu')
modelo_verificacion = PneumoniaCNN(num_classes=3)
modelo_verificacion.load_state_dict(checkpoint_cargado['state_dict'])
modelo_verificacion.eval()

tensor_prueba = torch.randn(1, 3, 100, 100)
with torch.no_grad():
    salida = modelo_verificacion(tensor_prueba)

print(f'Output shape: {list(salida.shape)}  ← esperado [1, 3]')
print('Modelo guardado y verificado correctamente.')